# Lab 2: Neural networks

In this lab, we will try our hands at building a small neural network from scratch to see how it works. The implementation is adapted from <a href="https://towardsdatascience.com/a-neural-network-from-scratch-c09fd2dea45d">https://towardsdatascience.com/a-neural-network-from-scratch-c09fd2dea45d</a>, which also contains some more background information and a different example.

## Basic implementation

Recall from the lecture that the basic element of a neural network is an artificial neuron, or node, which has weighted inputs that are summed and then fed into a non-linear activation function:

<img src="neuron.png">

Activation functions come in different variations:
<img src="activations.png">

And we can build a simple neural network by arranging multiple neurons in layers, and then connecting each neuron in one layer to each neuron in the next layer, including the input and output layers as well as hidden layers:
<img src="simplenn.png">
Let's start by implementing a class for such a fully-connected layer, sometimes also called a dense layer:

In [ ]:
import numpy as np

class DenseLayer:
    """
    A class to define fully connected layers.
    """
    
    def __init__(self, inputDimension, units, activation='', randomMultiplier=0.01):
        """
        Constructor:
          inputDimension: number of input features
          units: number of neurons in the layer
          activation: activation function applied to layer
            - options: 'sigmoid', 'tanh', 'relu', ''
          randomMultiplier: multiplier applied to the random weights during initialization
        """
        self.weights, self.bias = self.initialize(inputDimension, units, randomMultiplier)
        if activation == 'sigmoid':
            self.activation = activation
            self.activationForward = self.sigmoid
            self.activationBackward = self.sigmoidGrad
        elif activation == 'relu':
            self.activation = activation
            self.activationForward = self.relu
            self.activationBackward = self.reluGrad
        elif activation == 'tanh':
            self.activation = activation
            self.activationForward = self.tanh
            self.activationBackward = self.tanhGrad
        elif activation == 'softmax':
            self.activation = activation
            self.activationForward = self.softmax
            self.activationBackward = self.linear
        else:
            self.activation = 'none'
            self.activationForward = self.linear
            self.activationBackward = self.linear
    
    def initialize(self, nx, nh, randomMultiplier):
        """
        Initializes weights randomly:
          nx: number of input features
          nh: number of units
          randomMultiplier: multiplier applied to the random weights during initialization
        returns:
          weights: the randomly initialized weights
          bias: the bias terms
        """
        weights = randomMultiplier * np.random.randn(nh, nx)
        bias = np.zeros([nh, 1])
        return weights, bias


We also need methods for the activation functions. For some flexibility, we implement multiple ones plus their derivations:

In [ ]:
class DenseLayer(DenseLayer):
    
    def sigmoid(self, Z):
        """
        Sigmoid activation function
        """
        A = 1 / (1 + np.exp(-Z))
        return A

    def sigmoidGrad(self, dA):
        """
        Differential of sigmoid function with chain rule applied
        """
        s = 1 / (1 + np.exp(-self.prevZ))
        dZ = dA * s * (1 - s)
        return dZ


    def relu(self, Z):
        """
        Relu activation function
        """
        A = np.maximum(0, Z)
        return A

    def reluGrad(self, dA):
        """
        Differential of relu function with chain rule applied
        """
        s = np.maximum(0, self.prevZ)
        dZ = (s>0) * 1 * dA
        return dZ 


    def tanh(self, Z):
        """
        Tanh activation function
        """
        A = np.tanh(Z)
        return A

    def tanhGrad(self, dA):
        """
        Differential of tanh function with chain rule applied
        """
        s = np.tanh(self.prevZ)
        dZ = (1 - s**2) * dA
        return dZ

    def softmax(self, Z):
        exps = np.exp(Z - Z.max(axis=0))
        return exps / np.sum(exps, axis=0)
    
    def linear(self, Z):
        """
        Placeholder when no activation function is used
        """
        return Z

Now for the implementation of the calculation that a neuron performs - also called the forward step:

In [ ]:
class DenseLayer(DenseLayer):    
    def forward(self, A):
        """
        Forward pass through layer
          A: input vector
        """
        Z = np.dot(self.weights, A) + self.bias
        self.prevZ = Z
        self.prevA = A
        A = self.activationForward(Z)
        return A

Now recall how the network is trained - i.e. we update the weights of the neurons based on the gradient of the activation functions at the loss calculated on our ground truth data, so called gradient descent:
<img src="gradientdescent.png">

Let's implement the gradient calculation (the backward pass) as well as the weight update:

In [ ]:
class DenseLayer(DenseLayer):

    def backward(self, dA):
        """
        Backward pass through layer
          dA: previous gradient
        """
        dZ = self.activationBackward(dA)
        m = self.prevA.shape[1]
        self.dW = 1 / m * np.dot(dZ, self.prevA.T)
        self.db = 1 / m * np.sum(dZ, axis=1, keepdims=True)
        prevdA = np.dot(self.weights.T, dZ)
        return prevdA

    def update(self, learning_rate):
        """
        Update weights using gradients from backward pass
          learning_rate: the learning rate used in the gradient descent
        """
        self.weights = self.weights - learning_rate * self.dW
        self.bias = self.bias - learning_rate * self.db

Some helper methods for pretty printing:

In [ ]:
class DenseLayer(DenseLayer):

    def outputDimension(self):
        """
        Returns the output dimension for the next layer
        """
        return len(self.bias)

    
    def __repr__(self):
        """
        Used to print a pretty summary of the layer
        """
        act = 'none' if self.activation == '' else self.activation
        return f'Dense layer (nx={self.weights.shape[1]}, nh={self.weights.shape[0]}, activation={act})'

Next, we need a class to define our network itself - so far, without any layers:

In [ ]:
class NeuralNetwork:
    """
    Neural Network structure that holds our layers
    """
    
    def __init__(self, loss='cross-entropy', randomMultiplier = 0.01):
        """
        Constructor:
          loss: the loss function. Two are defined:
             - 'cross-entropy', 'categorical-cross-entropy', and 'mean-square-error'
          randomMultiplier: multiplier applied to the random weights during initialization
        """
        self.layers=[]
        self.randomMultiplier = randomMultiplier
        if loss=='cross-entropy':
            self.lossFunction = self.crossEntropyLoss
            self.lossBackward = self.crossEntropyLossGrad
        elif loss=='categorical-cross-entropy':
            self.lossFunction = self.categoricalCrossEntropyLoss
            self.lossBackward = self.categoricalCrossEntropyLossGrad
        elif loss=='mean-square-error':
            self.lossFunction = self.meanSquareError
            self.lossBackward = self.meanSquareErrorGrad
        self.loss=loss

How do we add a layer to the network? Let's create a method for that:

In [ ]:
class NeuralNetwork(NeuralNetwork): 
    def addLayer(self, inputDimension=None, units=1, activation=''):
        """
        Adds a Dense layer to the network:
          inputDimension: required when it is the first layer. otherwise takes dimensions of previous layer.
          units: number of neurons in the layer
          activation: activation function: valid choices are: 'sigmoid', 'tanh', 'relu', 'softma
        """
        if (inputDimension is None):
            inputDimension=self.layers[-1].outputDimension()
        layer = DenseLayer(inputDimension, units, activation, randomMultiplier= self.randomMultiplier)
        self.layers.append(layer)

And of course we need to define the loss of our network:

In [ ]:
class NeuralNetwork(NeuralNetwork): 
    
    def crossEntropyLoss(self, Y, A, epsilon=1e-15):
        """
        Cross Entropy loss function
          Y: true labels
          A: final activation function (predicted labels)
          epsilon: small value to make minimize chance for log(0) error
        """
        m = Y.shape[1]
        loss = -1 * (Y * np.log(A + epsilon) + (1 - Y) * np.log(1 - A + epsilon))
        cost = 1 / m * np.sum(loss)
        return np.squeeze(cost)
            
    def crossEntropyLossGrad(self, Y, A, epsilon=1e-15):
        """
        Cross Entropy loss Gradient
          Y: true labels
          A: final activation function (predicted labels)
        """
        dA = -(np.divide(Y, A+epsilon) - np.divide(1 - Y, 1 - A+epsilon))
        return dA
    
    def categoricalCrossEntropyLoss(self, Y, A, epsilon=1e-15):  
        m = Y.shape[1]
        loss = -1*(Y * np.log(A+epsilon))
        cost = 1 / m * (np.sum(loss)+epsilon)
        return np.squeeze(cost)
        
    
    def categoricalCrossEntropyLossGrad(self, Y, A, epsilon=1e-15):                
        dA = -np.divide(Y,A+epsilon)
        return(dA)
    
    def meanSquareError(self, Y, A):
        """
        Mean square error loss function
          Y: true labels
          A: final activation function (predicted labels)
        """
        loss = np.square(Y - A)
        m = Y.shape[1]
        cost = 1 / m * np.sum(loss)
        return np.squeeze(cost)
    
    def meanSquareErrorGrad(self, Y, A):
        """
        Mean square error loss gradient
          Y: true labels
          A: final activation function (predicted labels)
        """
        dA = -2 * (Y - A)
        return dA

    
    def cost(self, Y, A):
        """
        Cost function wrapper
          Y: true labels
          A: final activation function (predicted labels)
        """
        return self.lossFunction(Y, A)

Remember how the whole network is trained based on the gradient descent procedure within each layer? By propagating the gradient backwards throughout the whole network via the chain rule for derivations:
<img src="backprop.png">

Here is the implementation for the forward and backward passes as well as the weight updates for the whole network:

In [ ]:
class NeuralNetwork(NeuralNetwork): 

    def forward(self, X):
        """
        Forward pass through the whole model.
          X: input vector
        """
        x = np.copy(X)
        for layer in self.layers:
            x = layer.forward(x)
        return x
            
    
    def backward(self, A, Y):
        """
        backward pass through the whole model
          Y: true labels
          A: final activation function (predicted labels)
        """
        dA = self.lossBackward(Y, A)
        for layer in reversed(self.layers):
            dA = layer.backward(dA)
    
    
    def update(self, learning_rate=0.01):
        """
        Update weights and do a step of gradient descent for the whole model.
          learning_rate: learning_rate to use
        """
        for layer in self.layers:
            layer.update(learning_rate)

More helper functions:

In [ ]:
class NeuralNetwork(NeuralNetwork): 

    def __repr__(self):
        """
        Pretty print the model
        """
        layrepr = ['  ' + str(ix+1)+' -> ' + str(x) for ix, x in enumerate(self.layers)]
        return '[\n' + '\n'.join(layrepr) + '\n]'
   
    
    def numberOfParameters(self):
        """
        Print number of trainable parameters in the model
        """
        n = 0
        for layer in self.layers:
            n += np.size(layer.weights) + len(layer.bias)
        print(f'There are {n} trainable parameters in the model.')

## A simple regression example

We have everything we need for now. Let's try it out by creating a very simple network that learns to predict a linear function of each number that is put into it (i.e. linear regression of the form Y = aX+b).

First, let's create some training data. X are your training inputs while Y are the expected outputs:

In [ ]:
X = np.arange(-2, 5, 1).reshape([1, 7]) # Columns as examples
Y = #your code

Next, we create a simple network. As we have a linear problem, one layer should be enough. Consider which loss and activation function you need!

In [ ]:
np.random.seed(1)

#create network here

Now train your network for e.g. 1000 iterations. You will need the forward pass, the backward pass, and the cost updates. Make sure to regularly print the loss (= cost) to make sure your training is going correctly:

In [ ]:
#code for training your model

Finally, try out your model manually. Does it do what you expect it to do?

In [ ]:
model.forward(12)

## A classification example

Next, we will try our model for a more complex, non-linear problem.
To this end, we will try out a famous penguin data set that records various characteristics of individuals of three different species. There is more information here: <a href="https://github.com/rianrajagede/penguin-python">https://github.com/rianrajagede/penguin-python</a>.<br>
Let's load the data:

In [ ]:
import pandas as pd

#load
datatrain = pd.read_csv('penguins-clean-train.csv')

#Section 1.2 Preprocessing
from sklearn.preprocessing import StandardScaler

#change string value to numeric
datatrain.loc[datatrain['species']=='Adelie', 'species']=0
datatrain.loc[datatrain['species']=='Gentoo', 'species']=1
datatrain.loc[datatrain['species']=='Chinstrap', 'species']=2
datatrain = datatrain.apply(pd.to_numeric)
#uncomment this if you want to limit the data to two classes
#datatrain = datatrain[datatrain.species != 2]

#change dataframe to array
datatrain_array = datatrain.values

#split x and y (feature and target)
xtrain = datatrain_array[:,1:]
ytrain = datatrain_array[:,0]
#version 1: binary classification with one output neuron
#ytrain = np.expand_dims(ytrain, axis=0)
#version 2: three classes with on-hot representation
#ytrain = np.eye(3)[ytrain.astype(int)].T

#standardize
#palmer-penguin dataset has varying scales
scaler = StandardScaler()
xtrain = scaler.fit_transform(xtrain)
xtrain = xtrain.T


Now, implement your network. Once again, consider which output activation and loss you need. There are two options:
<ol>
    <li>Limit the data to two classes and implement this as a binary problem with one output node</li>
    <li>Use all three classes and implement three output nodes</li>
</ol>
Try out both! For the hidden layers, it is recommended to use ReLU activations. This is a relatively simply problem, so one hidden layer with e.g. 10 nodes may be sufficient.

In [ ]:
np.random.seed(1)

#create your network here

Once again, train your model and log the loss.
(Caveat: If your cost turns to 0, we may run into numerical issues with the current implementation. The easiest way to prevent this for now is to stop before it happens and adapt the learning rate).

In [ ]:
#...and train it

Compare your model output to the ground truth for some examples. You can also use the test data for this (see other CSV file).

In [ ]:
np.set_printoptions(suppress=True)
res = model.forward(xtrain[:,0:10])
print(res)
ytrain[:,0:10]

Feel free to try out the other variants or implement a validation step every 100 iterations or so!